# get similarity convergence

In [1]:
import os
import platform

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

In [2]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
research1_dir = os.getcwd()

# data/processed 폴더 위치 지정
processed_data_dir = research1_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')

# graph 이미지 저장할 폴더 위치 지정
graph_image_dir = research1_dir + ('\\graph\\slope' if os_system == 'Windows' else '/graph/slope')

In [3]:
# 폴더 없으면 생성
os.makedirs(graph_image_dir, exist_ok=True)

In [4]:
# 유사도, 유사도평균 coherence값을 저장한 테이블 읽어오기
tbl_data = pd.read_csv(processed_data_dir + 'similarity_coherence.csv', index_col=0, keep_default_na=False)
tbl_data[0:3]

,similarity_tear1_money,similarity_tear2_money,similarity_tear3_money,similarity_tear4_money,similarity_tear5_money,similarity_tear6_money,similarity_tear7_money,similarity_tear8_money,similarity_tear9_money,similarity_tear10_money,...,coherence_tear_relationships,coherence_family_relationships,coherence_mirror_relationships,coherence_abuse_relationships,coherence_relationships,coherence_tear_family,coherence_family_family,coherence_mirror_family,coherence_abuse_family,coherence_family
subject,,,,,,,,,,,,,,,,,,,,,
1,0.05485440967803956,0.11312033646465347,0.04415495198093822,0.08866880259195875,0.03045394742461427,0.12833422359085,0.10065367370467904,0.05435556682097242,0.1743477220561508,0.06917578590420992,...,0.10437839139747804,0.11159175646637291,0.03029209833241308,0.11065202713638661,0.08922856833316266,0.20934783739354912,0.12487956321975098,0.11293917958196284,0.17904907556024252,0.15655391393887635
2,0.05485440967803956,0.093303577702459,0.14638248146952804,0.10985657593682707,-0.011688823013538796,-0.01968727619281596,0.011553969605265668,0.11322701550540448,0.028949674231433464,0.13745887519736433,...,0.06098017328939255,0.05014490834752351,0.07189595857478095,0.052968230049086,0.05899731756519575,0.06944599572042942,0.10252325282708308,0.1554019398539225,0.10678515799590696,0.10853908659933548
3,-0.030720875019614624,0.09962618772051623,0.1332279900464325,-0.05457337174696808,0.0743048250412488,0.0710299168166334,0.03375842842242982,0.004347426250181918,0.1361629910326062,0.15753867694334311,...,0.07242303085693248,0.061945703535651195,0.06651371143000759,0.04626450524398483,0.06178673776664402,0.13093511750786943,0.1341830711968474,0.12230413679937391,0.10539788824477483,0.12320505343721638


In [5]:
def get_smoothed_similarity_func(similarity_df: pd.DataFrame, i_subject: int, topic: str):
    # trial 번호와 해당 trial에 대한 similarity 값을 배열로 변환
    trial_numbers = np.array(list(range(1, 41)))

    # 특정 피험자의 유사도 점수들 받아오기
    similarity_values = similarity_df.iloc[i_subject].tolist()
    similarity_values = np.array([float(value) if value != '' else 0.0 for value in similarity_values])

    # 데이터를 보간하는 함수 생성
    interpolation_function = interp1d(trial_numbers, similarity_values, kind='quadratic')

    # 정수값에 대한 데이터 추출
    integer_trial_numbers = np.arange(1, 41)
    integer_similarity_values = interpolation_function(integer_trial_numbers)

    # 부드러운 곡선을 위해 trial 번호를 더 자세히 나누기
    fine_trial_numbers = np.linspace(1, 40, 400)
    smoothed_similarity_values = interpolation_function(fine_trial_numbers)

    # 1차 함수 (선형 회귀)를 생성하여 예측값 얻기
    z = np.polyfit(integer_trial_numbers, integer_similarity_values, 1)
    p = np.poly1d(z)
    predicted_values = p(integer_trial_numbers)

    # 기울기와 절편 구하기
    slope = z[0]
    intercept = z[1]

    # 그래프 저장할 위치
    graph_image_path = (f'\\graph\\{topic}\\Subject_{i_subject}_similarity_curve.png' if os_system == 'Windows' else f'{graph_image_dir}/{topic}/Subject_{i_subject}_similarity_curve.png')
    # 폴더 없으면 생성
    os.makedirs(f'\\graph\\{topic}' if os_system == 'Windows' else f'{graph_image_dir}/{topic}', exist_ok=True)
    
    # 그래프 생성
    plt.figure(figsize=(10, 5))
    plt.plot(integer_trial_numbers, predicted_values, label='Linear Regression', color='g', linestyle='-')
    plt.plot(fine_trial_numbers, smoothed_similarity_values, label='Smoothed similarities', color='r')
    plt.scatter(integer_trial_numbers, integer_similarity_values, label='Real similarity Data', marker='o', color='b')
    plt.xlabel('Trial')
    plt.ylabel('Similarity Value')
    plt.title(f'Subject {i_subject}: Smoothed Similarity Curve')
    plt.legend()
    plt.grid(True)
    plt.savefig(graph_image_path)
    # plt.show()

    # 그래프 표시하지 않음
    plt.close()

    return predicted_values, (slope, intercept)


In [6]:
def add_convergence_values_to_df(start_index: int,end_index: int, dataframe: pd.DataFrame, seed_word: str, target_word: str):
    similarity_seed_target = tbl_data.iloc[:, start_index:end_index]
    slopes = []
    intercepts = []

    for i_subject in range(len(dataframe)):
        predicted_values, (slope, intercept) = get_smoothed_similarity_func(similarity_df = similarity_seed_target,
                                                                            i_subject = i_subject,
                                                                            topic = f'{seed_word}_{target_word}')
        slopes.append(slope)
        intercepts.append(intercept)
        
    # 모든 피험자의 convergence값을 얻은 후,
    tbl_data[f'convergence_slope_{seed_word}_{target_word}'] = slopes
    tbl_data[f'convergence_intercept_{seed_word}_{target_word}'] = intercepts

### target: money

In [14]:
target_word = 'money'

In [15]:

add_convergence_values_to_df(start_index=0,
                             end_index=40,
                             dataframe=tbl_data,
                             seed_word='tear',
                             target_word=target_word)
add_convergence_values_to_df(start_index=40,
                             end_index=80,
                             dataframe=tbl_data,
                             seed_word='family',
                             target_word=target_word)
add_convergence_values_to_df(start_index=80,
                             end_index=120,
                             dataframe=tbl_data,
                             seed_word='mirror',
                             target_word=target_word)
add_convergence_values_to_df(start_index=120,
                             end_index=160,
                             dataframe=tbl_data,
                             seed_word='abuse',
                             target_word=target_word)

### target: friend

In [16]:
target_word = 'friend'

In [17]:
add_convergence_values_to_df(start_index=160,
                             end_index=200,
                             dataframe=tbl_data,
                             seed_word='tear',
                             target_word=target_word)
add_convergence_values_to_df(start_index=200,
                             end_index=240,
                             dataframe=tbl_data,
                             seed_word='family',
                             target_word=target_word)
add_convergence_values_to_df(start_index=240,
                             end_index=280,
                             dataframe=tbl_data,
                             seed_word='mirror',
                             target_word=target_word)
add_convergence_values_to_df(start_index=280,
                             end_index=320,
                             dataframe=tbl_data,
                             seed_word='abuse',
                             target_word=target_word)

### target: relationships

In [18]:
target_word = 'relationships'

In [19]:
add_convergence_values_to_df(start_index=320,
                             end_index=360,
                             dataframe=tbl_data,
                             seed_word='tear',
                             target_word=target_word)
add_convergence_values_to_df(start_index=360,
                             end_index=400,
                             dataframe=tbl_data,
                             seed_word='family',
                             target_word=target_word)
add_convergence_values_to_df(start_index=400,
                             end_index=440,
                             dataframe=tbl_data,
                             seed_word='mirror',
                             target_word=target_word)
add_convergence_values_to_df(start_index=440,
                             end_index=480,
                             dataframe=tbl_data,
                             seed_word='abuse',
                             target_word=target_word)

### target: family

In [20]:
target_word = 'family'

In [21]:
add_convergence_values_to_df(start_index=480,
                             end_index=520,
                             dataframe=tbl_data,
                             seed_word='tear',
                             target_word=target_word)
add_convergence_values_to_df(start_index=520,
                             end_index=560,
                             dataframe=tbl_data,
                             seed_word='family',
                             target_word=target_word)
add_convergence_values_to_df(start_index=560,
                             end_index=600,
                             dataframe=tbl_data,
                             seed_word='mirror',
                             target_word=target_word)
add_convergence_values_to_df(start_index=600,
                             end_index=640,
                             dataframe=tbl_data,
                             seed_word='abuse',
                             target_word=target_word)

## save csv

In [22]:
tbl_data[0:3]

,similarity_tear1_money,similarity_tear2_money,similarity_tear3_money,similarity_tear4_money,similarity_tear5_money,similarity_tear6_money,similarity_tear7_money,similarity_tear8_money,similarity_tear9_money,similarity_tear10_money,...,convergence_slope_abuse_relationships,convergence_intercept_abuse_relationships,convergence_slope_tear_family,convergence_intercept_tear_family,convergence_slope_family_family,convergence_intercept_family_family,convergence_slope_mirror_family,convergence_intercept_mirror_family,convergence_slope_abuse_family,convergence_intercept_abuse_family
subject,,,,,,,,,,,,,,,,,,,,,
1,0.05485440967803956,0.11312033646465347,0.04415495198093822,0.08866880259195875,0.03045394742461427,0.12833422359085,0.10065367370467904,0.05435556682097242,0.1743477220561508,0.06917578590420992,...,-0.001506,0.133221,0.000234,0.204547,-0.000748,0.140217,0.003148,0.048414,-0.004465,0.257158
2,0.05485440967803956,0.093303577702459,0.14638248146952804,0.10985657593682707,-0.011688823013538796,-0.01968727619281596,0.011553969605265668,0.11322701550540448,0.028949674231433464,0.13745887519736433,...,0.000107,0.048129,-0.001593,0.100376,-0.002405,0.146696,-0.000017,0.151856,-0.002800,0.158849
3,-0.030720875019614624,0.09962618772051623,0.1332279900464325,-0.05457337174696808,0.0743048250412488,0.0710299168166334,0.03375842842242982,0.004347426250181918,0.1361629910326062,0.15753867694334311,...,-0.000438,0.055244,0.001286,0.098026,-0.000194,0.138164,-0.000334,0.129158,-0.003434,0.175805


In [23]:
# 단어 있는 버전 csv 저장
tbl_data.to_csv(processed_data_dir + 'convergence.csv')
